# No.2 1次元フーリエ変換 — 全課題パイプライン

課題1〜5の全フローを実行し、画像を保存します。

1. 矩形データ3種（DW=17,33,65）を生成
2. 各矩形データにFFTを適用
3. 矩形データをシフト（4通り）してプロット
4. シフト後の矩形データにFFTを適用
5. FFT後データをシフト→IFFTして再構成 + パワースペクトル

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import scipy.io
import os

DNUM = 1024
DW_list = [17, 33, 65]
shift_list = [-600, -300, 300, 600]
os.makedirs("output", exist_ok=True)

## 課題1: 矩形データ生成（DW=17, 33, 65）

In [ ]:
kukei_data = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, DW in enumerate(DW_list):
    Signal = np.zeros(DNUM, dtype=complex)
    d1 = DNUM // 2 - DW // 2
    d2 = DNUM // 2 + (DW + 1) // 2
    Signal[d1:d2] = 1.0
    kukei_data[DW] = Signal.copy()
    scipy.io.savemat(f"kukei_DW{DW}.mat", {"Signal": Signal})

    axes[idx].plot(Signal.real, linewidth=1, label="real part")
    axes[idx].plot(Signal.imag, linewidth=1, label="imaginary part")
    axes[idx].set_xlim(0, DNUM)
    axes[idx].set_title(f"kukei{DW}")
    axes[idx].set_xlabel("Data")
    axes[idx].set_ylabel("Amplitude")
    axes[idx].legend()
    axes[idx].grid(True)
fig.tight_layout()
fig.savefig("output/kadai1_kukei.png", dpi=150)
plt.show()

## 課題2: 各矩形データにFFTを適用

In [ ]:
fft_data = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, DW in enumerate(DW_list):
    Output = np.fft.fftshift(np.fft.fft(np.fft.fftshift(kukei_data[DW])))
    fft_data[DW] = Output.copy()
    scipy.io.savemat(f"kukei_DW{DW}_FFT.mat", {"Signal": Output})

    axes[idx].plot(Output.real, linewidth=1, label="real part")
    axes[idx].plot(Output.imag, linewidth=1, label="imaginary part")
    axes[idx].set_xlim(0, DNUM)
    axes[idx].set_title(f"kukei{DW}FFT")
    axes[idx].set_xlabel("Data")
    axes[idx].set_ylabel("Amplitude")
    axes[idx].legend()
    axes[idx].grid(True)
fig.tight_layout()
fig.savefig("output/kadai2_FFT.png", dpi=150)
plt.show()

## 課題3: 矩形データをシフト（4通り × 3種）

In [ ]:
shifted_kukei = {}
for DS in shift_list:
    DS_mod = DS % DNUM
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"シフト量 {DS}", fontsize=14)
    for idx, DW in enumerate(DW_list):
        S = np.roll(kukei_data[DW], DS_mod)
        shifted_kukei[(DW, DS)] = S.copy()
        name = f"kukei{DW}sft{DS_mod}"
        scipy.io.savemat(f"output/{name}.mat", {"Signal": S})

        axes[idx].plot(S.real, linewidth=1, label="real part")
        axes[idx].plot(S.imag, linewidth=1, label="imaginary part")
        axes[idx].set_xlim(0, DNUM)
        axes[idx].set_title(name)
        axes[idx].set_xlabel("Data")
        axes[idx].set_ylabel("Amplitude")
        axes[idx].legend()
        axes[idx].grid(True)
    fig.tight_layout()
    fig.savefig(f"output/kadai3_sft{DS_mod}.png", dpi=150)
    plt.show()

## 課題4: シフト後の矩形データにFFTを適用

In [ ]:
for DS in shift_list:
    DS_mod = DS % DNUM
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"シフト量 {DS} → FFT", fontsize=14)
    for idx, DW in enumerate(DW_list):
        S = shifted_kukei[(DW, DS)]
        F = np.fft.fftshift(np.fft.fft(np.fft.fftshift(S)))
        name = f"kukei{DW}sft{DS_mod}FFT"
        scipy.io.savemat(f"output/{name}.mat", {"Signal": F})

        axes[idx].plot(F.real, linewidth=1, label="real part")
        axes[idx].plot(F.imag, linewidth=1, label="imaginary part")
        axes[idx].set_xlim(0, DNUM)
        axes[idx].set_title(name)
        axes[idx].set_xlabel("Data")
        axes[idx].set_ylabel("Amplitude")
        axes[idx].legend()
        axes[idx].grid(True)
    fig.tight_layout()
    fig.savefig(f"output/kadai4_sft{DS_mod}FFT.png", dpi=150)
    plt.show()

## 課題5: FFT後データをシフト→IFFTして再構成 + パワースペクトル

FFT結果をシフトしてからIFFTする。周波数空間でのシフトが時間空間にどう影響するかを確認。
パワースペクトル $|F|=\sqrt{\mathrm{Re}^2+\mathrm{Im}^2}$ も比較。

In [ ]:
# FFT→シフト (実部・虚部プロット)
for DS in shift_list:
    DS_mod = DS % DNUM
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"FFT → シフト量 {DS}", fontsize=14)
    for idx, DW in enumerate(DW_list):
        F_shifted = np.roll(fft_data[DW], DS_mod)
        name = f"kukei{DW}FFTsft{DS_mod}"
        scipy.io.savemat(f"output/{name}.mat", {"Signal": F_shifted})

        axes[idx].plot(F_shifted.real, linewidth=1, label="real part")
        axes[idx].plot(F_shifted.imag, linewidth=1, label="imaginary part")
        axes[idx].set_xlim(0, DNUM)
        axes[idx].set_title(name)
        axes[idx].set_xlabel("Data")
        axes[idx].set_ylabel("Amplitude")
        axes[idx].legend()
        axes[idx].grid(True)
    fig.tight_layout()
    fig.savefig(f"output/kadai5_FFTsft{DS_mod}.png", dpi=150)
    plt.show()

In [ ]:
# FFT→シフト→IFFT (逆フーリエ変換で再構成)
for DS in shift_list:
    DS_mod = DS % DNUM
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"FFT → シフト量 {DS} → IFFT", fontsize=14)
    for idx, DW in enumerate(DW_list):
        F_shifted = np.roll(fft_data[DW], DS_mod)
        recon = np.fft.fftshift(np.fft.ifft(np.fft.fftshift(F_shifted)))
        name = f"kukei{DW}FFTsft{DS_mod}IFFT"

        axes[idx].plot(recon.real, linewidth=1, label="real part")
        axes[idx].plot(recon.imag, linewidth=1, label="imaginary part")
        axes[idx].set_xlim(0, DNUM)
        axes[idx].set_title(name)
        axes[idx].set_xlabel("Data")
        axes[idx].set_ylabel("Amplitude")
        axes[idx].legend()
        axes[idx].grid(True)
    fig.tight_layout()
    fig.savefig(f"output/kadai5_FFTsft{DS_mod}IFFT.png", dpi=150)
    plt.show()

### パワースペクトル比較

シフト量を変えてもパワースペクトル $|F| = \sqrt{\mathrm{Re}^2 + \mathrm{Im}^2}$ は変化しないことを確認。

In [ ]:
# パワースペクトル: シフト後の矩形データをFFTし、|F|を比較
for DW in DW_list:
    fig, ax = plt.subplots(figsize=(10, 4))
    # シフトなし
    ax.plot(np.abs(fft_data[DW]), linewidth=1, label="シフトなし", alpha=0.8)
    for DS in shift_list:
        DS_mod = DS % DNUM
        S = np.roll(kukei_data[DW], DS_mod)
        F = np.fft.fftshift(np.fft.fft(np.fft.fftshift(S)))
        ax.plot(np.abs(F), linewidth=1, label=f"sft{DS}", alpha=0.7)
    ax.set_xlim(0, DNUM)
    ax.set_title(f"パワースペクトル比較 (DW={DW})")
    ax.set_xlabel("Data")
    ax.set_ylabel("|F|")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    fig.savefig(f"output/kadai5_power_DW{DW}.png", dpi=150)
    plt.show()